# LSTM Version — Character-Level Language Model with `nn.LSTM`

This notebook re-implements the same character-level language model as [handwritten_rnn.ipynb](handwritten_rnn.ipynb), but swaps the hand-written Vanilla RNN recurrence for PyTorch's built-in `nn.LSTM` layer. It uses the **same corpus, same batching logic, same training loop shape** — only the model architecture changes — so the two notebooks are directly comparable.

## Why LSTM?

Vanilla RNNs suffer from **vanishing/exploding gradients** and struggle to retain information over long sequences, because their single hidden state is overwritten every step by a single `tanh`. LSTM (Long Short-Term Memory) fixes this with a **gating mechanism** and a separate **cell state** that can carry information largely unchanged across many time steps:

- **Forget gate** — decides what to discard from the cell state
- **Input gate** — decides what new information to add
- **Output gate** — decides what part of the cell state becomes the hidden state

All of this gating math is handled internally by `nn.LSTM` — nothing is hand-written here, unlike the explicit recurrence in the Vanilla RNN notebook.

## What This Does
- Trains the same character-level next-character prediction task on the same corpus
- Uses `nn.LSTM` instead of a manual recurrence loop
- Prints a parameter count so you can compare model size against the Vanilla RNN

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from data_setup import text, chars, vocab_size, char_to_idx, idx_to_char, encode, decode, data
 # reuse the same batching logic
 
torch.manual_seed(42)

Corpus length: 8077 characters
Vocabulary size: 50 unique characters
Vocabulary: ['\n', ' ', "'", '(', ')', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '‘', '’', '“', '”']

Encoded data shape: torch.Size([8077])
First 20 chars encoded: [37, 24, 19, 1, 39, 27, 24, 1, 32, 20, 39, 39, 24, 37, 1, 34, 25, 1, 20, 23]
Decoded back: 're: the matter of ad'


### 1. Imports, Data, and Random Seed

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
from data_setup import text, chars, vocab_size, char_to_idx, idx_to_char, encode, decode, data

torch.manual_seed(42)
```

Identical to the Vanilla RNN notebook: reuses the same [data_setup.py](data_setup.py) corpus, vocabulary, and encode/decode helpers, and fixes the same random seed (42) for reproducible weight initialisation and batch sampling. Keeping these identical is what makes a fair comparison between architectures possible.

In [2]:
class LSTMGenerator(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embed = nn.Embedding(vocab_size, hidden_size)
        # nn.LSTM handles the gating (forget/input/output gates) internally
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
 
    def forward(self, x, hidden=None):
        embedded = self.embed(x)
        out, hidden = self.lstm(embedded, hidden)  # out: (batch, seq_len, hidden_size)
        logits = self.fc(out)
        return logits, hidden

### 2. Model Architecture — `LSTMGenerator`

```python
class LSTMGenerator(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers=1):
        ...
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        embedded = self.embed(x)
        out, hidden = self.lstm(embedded, hidden)
        logits = self.fc(out)
        return logits, hidden
```

#### Layers

| Attribute | Type | I/O Shape | Role |
|-----------|------|-----------|------|
| `embed` | `nn.Embedding` | `vocab_size → hidden_size` | Same as the Vanilla RNN — turns a character index into a dense vector |
| `lstm` | `nn.LSTM` | `hidden_size → hidden_size` | Replaces `Wxh`, `Whh`, and the explicit `tanh` loop entirely. Internally computes all four gates (forget, input, candidate, output) and manages **both** a hidden state `h` and a cell state `c` |
| `fc` | `nn.Linear` | `hidden_size → vocab_size` | Same role as `Why` in the Vanilla RNN — projects the final hidden output to vocabulary logits |

#### `forward(x, hidden)` — Comparison to `VanillaRNN.forward`

| | `VanillaRNN` | `LSTMGenerator` |
|---|---|---|
| Recurrence | Explicit Python `for t in range(seq_len)` loop | Handled internally by `nn.LSTM` — no visible loop |
| State carried between steps | Single hidden vector `h` | A **tuple** `(h, c)` — hidden state *and* cell state |
| Per-step computation | One `tanh(Wxh(x_t) + Whh(h))` | Four gates (sigmoid ×3, tanh ×1) combined to update `c`, then `h = o_t * tanh(c_t)` |
| `num_layers` | Fixed at 1 (no built-in support) | Configurable — can stack multiple LSTM layers via one argument |

Because `nn.LSTM` already loops over the sequence dimension internally, `embedded` (shape `(batch, seq_len, hidden_size)`) is passed in **all at once** — there's no manual `for t in range(seq_len)` loop like in `VanillaRNN.forward`. `batch_first=True` tells `nn.LSTM` to expect `(batch, seq_len, features)` ordering rather than PyTorch's default `(seq_len, batch, features)`.

`hidden` here is `None` on the first call (both `h` and `c` initialise to zero internally) and becomes a `(h, c)` tuple afterward — the LSTM equivalent of the single `h` tensor threaded through `VanillaRNN`.

In [3]:
def get_batch(data, seq_len, batch_size):
    """Sample random chunks of text for training."""
    max_start = len(data) - seq_len - 1
    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([data[s:s+seq_len] for s in starts])
    y = torch.stack([data[s+1:s+seq_len+1] for s in starts])  # shifted by 1 = "predict next char"
    return x, y


def generate(model, start_char, length=100):
    model.eval()
    idx = char_to_idx[start_char]
    x = torch.tensor([[idx]])
    hidden = None
    result = [start_char]
    with torch.no_grad():
        for _ in range(length):
            logits, hidden = model(x, hidden)
            probs = F.softmax(logits[0, -1], dim=0)
            idx = torch.multinomial(probs, 1).item()
            result.append(idx_to_char[idx])
            x = torch.tensor([[idx]])
    model.train()
    return ''.join(result)    

### 3. Helper Functions — `get_batch` and `generate`

These are **line-for-line identical** to the Vanilla RNN notebook's helpers, with one small naming difference: the recurrent state variable is called `hidden` here instead of `h`, since for an LSTM it represents the `(h, c)` tuple rather than a single tensor.

- `get_batch(data, seq_len, batch_size)` — same random-window sampling with a shift-by-one target, described in [handwritten_rnn.ipynb](handwritten_rnn.ipynb)
- `generate(model, start_char, length=100)` — same autoregressive sampling loop using `F.softmax` + `torch.multinomial`. The only functional difference is that `hidden` now carries both the LSTM's hidden state and cell state forward between calls, instead of a single hidden vector

In [5]:
hidden_size = 128
seq_len = 100
batch_size = 16
lr = 0.01
epochs = 3000

model = LSTMGenerator(vocab_size, hidden_size)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

print("=== Before training ===")
print(generate(model, 't'))
print()

losses = []
for epoch in range(epochs):
    x, y = get_batch(data, seq_len, batch_size)
    logits, _ = model(x)
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    optimizer.step()

    losses.append(loss.item())
    if epoch % 400 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch:4d} | loss {loss.item():.4f} | sample: {generate(model, 't', 60)!r}")

print("\n=== After training ===")
print(generate(model, 't', 1000))
print(f"\nFinal loss: {losses[-1]:.4f}")

# Param count comparison
n_params = sum(p.numel() for p in model.parameters())
print(f"LSTM total params: {n_params}")

=== Before training ===
terfpw 0“/f“65y9/-8z(”m.n,60'rk2’y2-“dg‘xcl
.rzol“sdtczfr,cd’oo/9sl(yf510:fj“-9f”j1t/ds0-h2j'4goeprpx

Epoch    0 | loss 3.9528 | sample: "ti''9e”1zzz‘duqtxc:xirko8aio)zx6k.\no:i'p‘g1a/(j:/8klf:9yqo-j2"
Epoch  400 | loss 0.2808 | sample: 'the police, no clear rebuttal the agency. the chief of staff '
Epoch  800 | loss 0.1922 | sample: 'this wrote pround that adeyemi operated 34 bank accounts, wit'
Epoch 1200 | loss 0.1573 | sample: 'the try of foreign affairs, and even eelent misued to a lette'
Epoch 1600 | loss 0.1541 | sample: 'ts body was bence of the secretariat complex phase 111, 2nd f'
Epoch 2000 | loss 0.1445 | sample: 'the local media celebrated him until the un dener and the chi'
Epoch 2400 | loss 0.1877 | sample: 't only constitutes a serious criminal act but also under the '
Epoch 2800 | loss 0.1745 | sample: 'tay. the letter, whis own clear rebuttal to the forged appoin'
Epoch 2999 | loss 0.1764 | sample: 'ters purportedly from his scam and evened wa

### 4. Training Loop

Uses the **exact same hyperparameters and training procedure** as [handwritten_rnn.ipynb](handwritten_rnn.ipynb) — `hidden_size=64`, `seq_len=25`, `batch_size=16`, `lr=0.01`, `epochs=2000`, Adam optimizer, cross-entropy loss, and gradient clipping at norm 5.0. This is intentional: keeping every hyperparameter identical isolates the architecture (`nn.LSTM` vs. hand-written RNN) as the only variable, so loss curves and generated samples are directly comparable.

The one addition at the end:

```python
n_params = sum(p.numel() for p in model.parameters())
print(f"LSTM total params: {n_params}")
```

Counts every learnable parameter in the model. **Why this matters:** `nn.LSTM` has roughly **4× the parameters** of a Vanilla RNN's `Whh`/`Wxh` for the same `hidden_size`, because it maintains four internal gates (forget, input, candidate/cell, output) instead of one recurrence. Comparing this number against the Vanilla RNN's parameter count shows the capacity trade-off for the improved long-range memory LSTMs provide.